In [1]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sha2, when
from pyspark.sql import SparkSession

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
products_stg_df = spark.read.parquet("/user/sclsepb4532/sclsepb4532_staging.db/products_stg")

26/11/08 14:36:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 14:36:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/08 14:36:19 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/08 14:36:21 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/11/08 14:37:42 WARN DataStreamer: Exception for BP-1364978691-172.31.64.10-1677754292516:blk_1101628350_27896246
java.net.SocketTimeoutException: 75000 millis timeout while waiting for channel to be ready for read. ch : java.nio.channels.SocketChannel[connected local=/172.31.8.183:57026 remote=/172.31.64.13:9866]
	at org.apache.hadoop.net.SocketIOWithTimeout.doIO(SocketIOWithTimeout.

In [2]:
dim_product_df = products_stg_df.select(
    sha2(col("product_id"), 256).alias("product_key"),
    col("category"),
    col("brand"),
    when(col("price") < 1000.00, "Low")
    .when((col("price") <= 5000.00), "Medium")
    .otherwise("Premium").alias("price_band"))

In [3]:
dim_product_df.write\
.mode("overwrite")\
.option("compression", "snappy")\
.saveAsTable("sclsepb4532_analytics.dim_product_analytics")

26/11/08 14:40:18 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
